## Imports
This can take awhile. Upwards of 3 minutes on an A100 Node

NOTE: Do not "run all". I've observed a consistent crash if you run past the cell defining the policy. Run until that cell, and then you can choose which evaluation you want to do.

In [1]:
import math
import os
import sys
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".5"

import numpy as np
import mediapy as media

import robosuite
from robosuite.controllers import load_composite_controller_config

from openpi.models import model as _model
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config

sys.path.append('../py_script')
import profiling as prof
prof.prof_start()

[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /home/jcpeng24/workspace/mujoco_test/mujoco_playground/.venv/lib/python3.12/site-packages/robosuite/scripts/setup_macros.py (macros.py:59)
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)


## Start ollama
Start ollama as a background process.
This can cause problems: If you want to interrupt the script, you will have to re-run the following cell.

In [2]:
import subprocess
import time
ollama_logfile = open("ollama.log", 'w')
_ollama_shell = subprocess.Popen(["bash", "-c", "module load ollama/0.9.0; ollama serve"], stdin=subprocess.DEVNULL, stdout=ollama_logfile, stderr=ollama_logfile)
time.sleep(1) # TODO: actually wait

from ollama import Client
import requests
from tool_api import ToolHandler
from agent_responses import tool_wrap, StringResponse, ImageResponse

base_url="http://localhost:11434"  # Running locally
#context_length = 120000
model_name="gemma3:27b"
context_length = 2400

if requests.get(base_url).status_code == 200:
    client = Client(host=base_url)
    api_return = client.list()
    avail_models = [model['model'] for model in api_return['models']]
else:
    print("Ollama is not running")
    exit(1)

if model_name not in avail_models:
    print(f"Model {model_name} is not available (out of {avail_models})")
    exit(2)

In [3]:
# Forcibly spin up ollama, define chat function
tool_handler = ToolHandler()
tool_handler.reset_chat("You are an AI assistant controlling a robot.")

def chat(user_message):
    tool_handler.message_state.append({
        'role': 'user', 'content': user_message
    })
    return tool_handler.tool_chat(client,
            model=model_name,
            keep_alive=-1,
            options=dict(
                num_ctx=context_length,
                temperature=0.0
            )
        )
chat("Hello!")

At each turn, if you decide to invoke any of the function(s), it should be wrapped with ```tool_code```. The python methods described below are imported and available, you can only use defined methods. The generated code should be readable and efficient. The response to a method will be wrapped in ```tool_output``` use it to call more tools or generate a helpful, friendly response. When using a ```tool_call``` think step by step why and how it should be used.
 
The following Python methods are available:
 
```python

```

{'role': 'user', 'content': 'Hello!'}
??? no content in chunk



"Hello there! I'm here and ready to assist. How can I help you today? Perhaps you'd like me to perform a task with the robot, or maybe you just want to chat. Let me know what's on your mind."

In [4]:
def _quat2axisangle(quat):
    """
    Copied from robosuite: https://github.com/ARISE-Initiative/robosuite/blob/eafb81f54ffc104f905ee48a16bb15f059176ad3/robosuite/utils/transform_utils.py#L490C1-L512C55
    """
    # clip quaternion
    if quat[3] > 1.0:
        quat[3] = 1.0
    elif quat[3] < -1.0:
        quat[3] = -1.0

    den = np.sqrt(1.0 - quat[3] * quat[3])
    if math.isclose(den, 0.0):
        # This is (close to) a zero degree rotation, immediately return
        return np.zeros(3)

    return (quat[:3] * 2.0 * math.acos(quat[3])) / den

## Controller and environment definition

Joint control (fine tuned on droid dataset), or EE position (libero dataset).

Joint control hasn't been fully tested with the subtask generation pipeline. (It would likely yield garbage results anyway, just like the finetuned libero model.)

In [5]:
# Joint control
# controller_file = os.path.join(os.path.dirname(__file__), "..", "py_script", "panda_joint_controller.json")
# controller_config = load_composite_controller_config(controller=controller_file)
# Load pi model
# vla_config = _config.get_config("pi05_droid")
# checkpoint_dir = download.maybe_download("gs://openpi-assets/checkpoints/pi05_droid")
# @prof.profiled
# def prompt_from_obs(obs, prompt):
#    qpos = np.array(obs['robot0_joint_pos'])
#    gripper_pos = np.array(obs['robot0_gripper_qpos'][0])
#    return {
#        # Flip the ego camera?
#        'observation/exterior_image_1_left': obs['agentview_image'][::-1, ::-1, :],
#        'observation/wrist_image_left': obs['robot0_eye_in_hand_image'][::-1, ::-1, :],
#        'observation/joint_position': qpos,
#        'observation/gripper_position': gripper_pos,
#        'prompt': prompt
#    }

In [6]:
# Pose control
controller_config = load_composite_controller_config(controller="BASIC")
# Load pi model
vla_config = _config.get_config("pi05_libero")
checkpoint_dir = download.maybe_download("gs://openpi-assets/checkpoints/pi05_libero")
@prof.profiled
def prompt_from_obs(obs, prompt):
    # Reference: https://github.com/Physical-Intelligence/openpi/blob/981483dca0fd9acba698fea00aa6e52d56a66c58/examples/libero/main.py#L130
    return {
        # Flip the ego camera?
        'observation/image': obs['agentview_image'][::-1, ::-1, :],
        'observation/wrist_image': obs['robot0_eye_in_hand_image'][::-1, ::-1, :],
        'observation/state': np.concatenate(
            (
                obs["robot0_eef_pos"],
                _quat2axisangle(obs["robot0_eef_quat"]),
                obs["robot0_gripper_qpos"],
            )
        ),
        'prompt': prompt
    }

[robosuite INFO] Loading controller configuration from: /home/jcpeng24/workspace/mujoco_test/mujoco_playground/.venv/lib/python3.12/site-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)
INFO:robosuite_logs:Loading controller configuration from: /home/jcpeng24/workspace/mujoco_test/mujoco_playground/.venv/lib/python3.12/site-packages/robosuite/controllers/config/default/composite/basic.json


In [7]:
freq = 20
episode_length=800
env = robosuite.make(
    "Stack",
    robots=["Panda"],
    gripper_types="default",
    controller_configs=controller_config,
    env_configuration="opposed",    # What are the options?
    has_renderer=False,
    #render_camera="frontview",
    has_offscreen_renderer=True,
    control_freq=freq,
    horizon=episode_length,
    use_object_obs=False,
    use_camera_obs=True,
    camera_names=["agentview", "robot0_eye_in_hand"],
    camera_heights=224,
    camera_widths=224,
)
env_step = prof.profiled(env.step, name="step")

# Create a trained policy.
policy = _policy_config.create_trained_policy(vla_config, checkpoint_dir)
# NOTE: DO NOT RUN PAST THIS CELL WHEN STARTING UP! IT CAUSES A ASYNCIO CRASH FOR SOME REASON

[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

  0%|          | 0.00/4.07M [00:00<?, ?iB/s]

# STOP HERE WHEN RUNNING AND WAIT FOR ABOVE CELL TO FINISH!

### Define data structures for collecting outputs

Run this cell to clear collected data buffers.

In [12]:
rollout = []
frames = []
wrist_frames = []
snapshots = []
statuses = []
subtasks = []
act_scale = 0.5

### Collect output and video for one run.

Don't run this if you want to run the VLM verification things.

In [9]:
for _ in range(1):
    obs = env.reset()
    prompt = prompt_from_obs(obs)
    vla_output = policy.infer(prompt)
    actions = vla_output['actions']
    print(vla_output.get('subtask', 'no subtask output'), flush=True)
    rollout.append(obs)

    policy_infer = prof.profiled(policy.infer, name="infer")

    trajectory_idx = 0
    for i in range(episode_length):
        act = np.copy(actions[trajectory_idx])
        act[-1] *= 1
        obs, reward, done, info = env_step(act * act_scale)
        rollout.append(obs)
        frames.append(obs['agentview_image'][::-1, ::-1, :])
        wrist_frames.append(obs['robot0_eye_in_hand_image'][::-1, ::-1, :])
        trajectory_idx += 1
        if trajectory_idx == (len(actions)//2):
            prompt = prompt_from_obs(obs)
            vla_output = policy_infer(prompt)
            actions = vla_output['actions']
            print(vla_output.get('subtask', 'no subtask output'), flush=True)
            trajectory_idx = 0

media.write_video(f'franka_stacking.mp4', frames, fps=freq)
media.write_video(f'franka_stacking_wrist.mp4', wrist_frames, fps=freq)

[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

TypeError: prompt_from_obs() missing 1 required positional argument: 'prompt'

### Running with VLM verification in the loop

Not exactly like Verytrace.
The outer loop (plan loop) is being handled completely by one instance of Gemma3, which is making a tool call to command the VLA to take an action (`execute_action` function.).
The planner VLM is also told to explain its plan through the arguments to the function.
The simulator runs for a fixed number of steps, then the verifier VLM is given the function arguments and final robot state (as an image).
The verifier checks if the intent is reflected in the image, and returns the result (SUCCESS, IN_PROGRESS, or FAILURE) to the planner.

This first cell can take a long time to run, since it jit compiles the inference function.

In [10]:
obs = env.reset()
prompt = prompt_from_obs(obs, "Don't move")
vla_output = policy.infer(prompt)
actions = vla_output['actions']
observed_image = obs['agentview_image'][::-1, ::-1, :]
policy_infer = prof.profiled(policy.infer, name="infer")

@tool_wrap(ImageResponse)
def get_robot_image() -> ImageResponse:
    """
    Get the image the robot is currently seeing.
    Use this at the start of a task, to understand what the camera sees.
    """
    return observed_image

def verify(image, action, postcondition):
    verifier_prompt = """
    You are a careful AI assistant auditing a robot. The robot was told to perform a series of actions,
    but it is error prone. One action in this series is shown to you, intended to complete a subtask.
    Your job is to decide if the action was successfully executed or not.

    You should return a json string, of the form:

    ```json
    {
      "status": ["OK"|"IN_PROGRESS"|"FAILURE"],
      "reason": "put reasoning here"
    }
    ```

    Status should be:
     - OK if the execution succeeded fully.
     - IN_PROGRESS if it looks like the execution is on track to succeed.
       For example, if the subtask is to pick up an object and the robot is near the object but not grasped yet.
     - FAILURE if the subtask failed, for example the robot picked up the wrong object.

    Also supply the reason for your decision.
    """
    messages = [
        {'role': 'system', 'content': verifier_prompt},
        {
            'role': 'user',
            'content': f"The action was: {action}, the subgoal is: {postcondition}",
            'images': [ImageResponse.encode_image(image)]
        }
    ]
    while True:
        resp = client.chat(model=model_name, stream=False, messages=messages)
        resp_str = resp.message.content
        if "```json" not in resp_str:
            print("Verification failed... retry")
            messages.append({'role': 'user', 'content': "I don't see an answer..."})
            continue
        end = resp_str.split('```json', 1)[1].replace('```', '').strip()
        print("Verification result:", end)
        return end

#@tool_wrap(ImageResponse)
def execute_action(action: str, postcondition: str) -> ImageResponse:
    """
    Attempt to execute a short action from the robot.

    Arguments:
        action: The action to execute. This should be a short, "single step" action.
        postcondition: Condition to be satisfied at the end of this action. For example, "robot is holding cup"

    Return: New image of the scene after robot has attempted action execution, along with status message (sucess or failure)
    """
    global obs, observed_image
    
    prompt = prompt_from_obs(obs, action)
    vla_output = policy.infer(prompt)
    actions = vla_output['actions']
    #print(vla_output.get('subtask', 'no subtask output'), flush=True)

    trajectory_idx = 0
    for i in range(100):
        act = np.copy(actions[trajectory_idx])
        act[-1] *= 1
        obs, reward, done, info = env_step(act * act_scale)
        prompt = prompt_from_obs(obs, action)
        rollout.append(prompt['observation/state'])
        frames.append(obs['agentview_image'][::-1, ::-1, :])
        wrist_frames.append(obs['robot0_eye_in_hand_image'][::-1, ::-1, :])
        trajectory_idx += 1
        if trajectory_idx == (len(actions)//2):
            vla_output = policy_infer(prompt)
            actions = vla_output['actions']
            #print(vla_output.get('subtask', 'no subtask output'), flush=True)
            trajectory_idx = 0
    observed_image = frames[-1]
    snapshots.append(observed_image)
    msg = verify(observed_image, action, postcondition)
    statuses.append(msg)
    subtasks.append((action, postcondition))
    return ImageResponse(img=observed_image, message=msg)

tool_handler = ToolHandler()
tool_handler.register(get_robot_image)
tool_handler.register(execute_action)

llm_prompt = """
You are a careful AI assistant controlling a robot using natural language.
You have control over a rudimentary low-level controller that can be comamnded by short natural language commands.
When given a task, you should break it down into steps, and use `execute_action` to execute the steps one by one,
checking that the execution succeeded by reading its return output.

If the execution did not succeed, try to interpret the error message given. Sometimes, you may just need to
execute the same task again (for longer tasks); sometimes you may need to try a different approach.
For example, if the robot picked up the wrong object, you may have to tell it to put down the object it's holding
before picking up the correct object. Or, you can try identifying the object in different ways, such as by position, size,
or color.

If you get an IN_PROGRESS status, you should reissue the same command, until a success or failure status is read.
If the episode terminated, don't try to restart it. Just tell the user you couldn't complete the task and terminate.

When you are done with the task, simply tell the user they are done by emitting the string "Done".
"""

[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

### Manually running the "verified" loop

Pretty easy, LLM handles tool calling.
Next cell is result visualization

In [ ]:
tool_handler.reset_chat(llm_prompt)
task = "Stack the red cube onto the green cube."
s = chat(task)
print("Final output:", s)

In [ ]:
import matplotlib.pyplot as plt
print(s)
idx = 0
plt.figure(0)
plt.clf()
plt.imshow(snapshots[idx])
print(statuses[idx])
print(subtasks[idx])

### Run multiple trials and save the results

Result videos, and observations/verifier and planner traces in a pickle file

In [15]:
import pickle

def run_one_trial(trial_idx):
    global obs, observed_image
    global rollout, frames, wrist_frames, snapshots, statuses, subtasks
    obs = env.reset()
    observed_image = obs['agentview_image'][::-1, ::-1, :]
    tool_handler.reset_chat(llm_prompt)
    task = "Stack the red cube onto the green cube."
    s = chat(task)
    frames.append(observed_image)
    wrist_frames.append(obs['robot0_eye_in_hand_image'][::-1, ::-1, :])

    with open(f"output/franka_stacking.{trial_idx}.pkl", "wb") as outfile:
        pickle.dump({
            "snapshots": snapshots,
            "statuses": statuses,
            "subtasks": subtasks,
            "state_traj": rollout
        }, outfile)
    media.write_video(f'output/franka_stacking.{trial_idx}.mp4', frames, fps=freq)
    media.write_video(f'output/franka_stacking_wrist.{trial_idx}.mp4', wrist_frames, fps=freq)
    
    rollout = []
    frames = []
    wrist_frames = []
    snapshots = []
    statuses = []
    subtasks = []

In [ ]:
rollout = []
frames = []
wrist_frames = []
snapshots = []
statuses = []
subtasks = []
act_scale = 0.5
for i in range(60):
    print(f"Running trial {i}")
    run_one_trial(i)

In [17]:
# Profiling information
prof.prof_dump_info()

Total time: 90083.53065061569
> Function prompt_from_obs
           |       Time |    Calls |     Average |         Min |         Max | Thread Info
           |    1.385157 |   33054 |  0.00004191 |  0.00002529 |  0.00043238 | MainThread
<< Totals: |  1.38515659 |   33054 |  0.00004191 |  0.00002529 |  0.00043238 | 1 threads
> Function step
           |       Time |    Calls |     Average |         Min |         Max | Thread Info
           | 4771.955290 |   32700 |  0.14593135 |  0.01487168 |  9.81596894 | MainThread
<< Totals: | 4771.95528968 |   32700 |  0.14593135 |  0.01487168 |  9.81596894 | 1 threads
> Function infer
           |       Time |    Calls |     Average |         Min |         Max | Thread Info
           | 1541.363118 |    6540 |  0.23568243 |  0.22788087 |  0.30690147 | MainThread
<< Totals: | 1541.36311843 |    6540 |  0.23568243 |  0.22788087 |  0.30690147 | 1 threads


In [ ]:
# Manual chat
client.chat(model=model_name, messages=[{'role': 'user', 'content': 'hello'}], stream=False)